# Prime Numbers Lab — Notebook 18

## Spectral memory decomposition and low-rank correction

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Notebook:** `18_spectral_memory_decomposition_and_low_rank_correction.ipynb`  
**Core question:** Can the two-step residue transition memory measured in Notebook 17 be compressed into a low-rank correction to the first-order Markov baseline?

**Core model:**

`P2_empirical = P^2 + Delta ≈ P^2 + M_k`

Where:
- `P` is the first-order transition operator.
- `P2_empirical` is the observed two-step operator.
- `Delta` is the memory operator.
- `M_k` is a rank-k approximation of Delta.

**Template lock:** flat output files, numbered filenames, manifest CSV, summary markdown, optional Colab download cell.


## 0. Setup

This notebook is standalone: it recomputes prime residue transitions from scratch, while preserving the same output pattern used in the previous notebooks.

In [ ]:
import os, math, zipfile, json, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "18"
NOTEBOOK_SLUG = "spectral_memory_decomposition_and_low_rank_correction"
OUTPUT_ZIP = f"{NOTEBOOK_ID}_{NOTEBOOK_SLUG}_outputs.zip"

np.random.seed(9423)

# Flat-output template standard: write all artifacts to current working directory.
OUT = Path(".")
manifest = []

def savefig(name, dpi=160):
    path = OUT / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    manifest.append({"file": name, "type": "figure"})
    return path

def save_csv(df, name):
    path = OUT / name
    df.to_csv(path, index=False)
    manifest.append({"file": name, "type": "data"})
    return path

def save_text(text, name):
    path = OUT / name
    path.write_text(text, encoding="utf-8")
    manifest.append({"file": name, "type": "text"})
    return path

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "font.size": 12,
})


## 1. Generate prime transition data

We use the eight admissible residue states modulo 30: `1, 7, 11, 13, 17, 19, 23, 29`.

The sequence of prime residues becomes a finite symbolic trajectory. Notebook 18 studies first-order and two-step transition operators on that trajectory.

In [ ]:
LIMIT = 1_500_000
RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
STATE_INDEX = {int(r): i for i, r in enumerate(RESIDUES)}
N_STATES = len(RESIDUES)

def sieve_primes(n):
    """Return primes <= n using a vectorized sieve."""
    if n < 2:
        return np.array([], dtype=np.int64)
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    limit = int(n**0.5) + 1
    for p in range(2, limit):
        if sieve[p]:
            sieve[p*p:n+1:p] = False
    return np.flatnonzero(sieve).astype(np.int64)

primes_all = sieve_primes(LIMIT)
primes = primes_all[primes_all > 5]
residue_values = primes % 30
states = np.array([STATE_INDEX[int(r)] for r in residue_values], dtype=int)

gaps = np.diff(primes)
x_mid = primes[:-1]
z = gaps / np.log(x_mid)

summary = {
    "limit": LIMIT,
    "prime_count_total": int(len(primes_all)),
    "prime_count_used": int(len(primes)),
    "transition_count": int(len(states) - 1),
    "two_step_transition_count": int(len(states) - 2),
    "state_count": int(N_STATES),
}
summary


## 2. Build first-order and empirical two-step operators

We compute first-order transition probabilities, empirical two-step transition probabilities, the Markov baseline `P @ P`, and the memory delta `Delta = P2_empirical - P2_markov`.

In [ ]:
def row_normalize(counts):
    counts = np.asarray(counts, dtype=float)
    row_sums = counts.sum(axis=1, keepdims=True)
    return np.divide(counts, row_sums, out=np.zeros_like(counts), where=row_sums > 0)

C1 = np.zeros((N_STATES, N_STATES), dtype=float)
for a, b in zip(states[:-1], states[1:]):
    C1[a, b] += 1
P = row_normalize(C1)

C2 = np.zeros((N_STATES, N_STATES), dtype=float)
for a, c in zip(states[:-2], states[2:]):
    C2[a, c] += 1
P2_empirical = row_normalize(C2)
P2_markov = P @ P
Delta = P2_empirical - P2_markov

row_check = pd.DataFrame({
    "state": RESIDUES,
    "P_row_sum": P.sum(axis=1),
    "P2_empirical_row_sum": P2_empirical.sum(axis=1),
    "P2_markov_row_sum": P2_markov.sum(axis=1),
    "delta_row_sum": Delta.sum(axis=1),
})
save_csv(row_check, "18_row_sum_checks.csv")
row_check


In [ ]:
def matrix_df(M):
    return pd.DataFrame(M, index=[f"from_{r}" for r in RESIDUES], columns=[f"to_{r}" for r in RESIDUES]).reset_index(names="state")

save_csv(matrix_df(P), "18_transition_operator_P.csv")
save_csv(matrix_df(P2_empirical), "18_empirical_two_step_operator_P2.csv")
save_csv(matrix_df(P2_markov), "18_markov_predicted_two_step_operator_P2.csv")
save_csv(matrix_df(Delta), "18_two_step_memory_delta_operator.csv")


## 3. Visualize operators

These heatmaps repeat Notebook 16/17 visual grammar: first-order operator, empirical two-step operator, Markov-predicted two-step operator, and memory delta.

In [ ]:
def heatmap(M, title, cbar_label, fname, cmap="viridis", center_zero=False):
    plt.figure(figsize=(8, 7))
    if center_zero:
        vmax = np.max(np.abs(M))
        im = plt.imshow(M, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    else:
        im = plt.imshow(M, aspect="auto", cmap=cmap)
    plt.colorbar(im, label=cbar_label)
    plt.xticks(range(N_STATES), RESIDUES)
    plt.yticks(range(N_STATES), RESIDUES)
    plt.xlabel("next state")
    plt.ylabel("current state")
    plt.title(title)
    savefig(fname)

heatmap(P, "First-order transition operator P", "P(s_next | s)", "18_transition_operator_P_heatmap.png")
heatmap(P2_empirical, "Empirical two-step operator", "P(s_{n+2} | s_n)", "18_empirical_two_step_operator_heatmap.png")
heatmap(P2_markov, "Markov predicted two-step operator P²", "P²(s_{n+2} | s_n)", "18_markov_predicted_two_step_operator_heatmap.png")
heatmap(Delta, "Memory delta: empirical P² minus Markov P²", "delta probability", "18_two_step_memory_delta_heatmap.png", center_zero=True)


## 4. Spectral decomposition of P

Eigenvalues of `P` reveal mixing structure. The spectral-gap proxy is `1 - abs(lambda_2)`.

In [ ]:
eigvals, eigvecs = np.linalg.eig(P.T)
order = np.argsort(-np.abs(eigvals))
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

lambda_abs = np.abs(eigvals)
spectral_gap = float(1 - lambda_abs[1]) if len(lambda_abs) > 1 else np.nan
mixing_proxy = float(1 / spectral_gap) if spectral_gap > 0 else np.inf

stationary = np.real(eigvecs[:, 0])
stationary = np.abs(stationary)
stationary = stationary / stationary.sum()

spectral_metrics = pd.DataFrame({
    "metric": ["lambda_1_abs", "lambda_2_abs", "spectral_gap", "mixing_proxy_1_over_gap"],
    "value": [float(lambda_abs[0]), float(lambda_abs[1]), spectral_gap, mixing_proxy],
})
save_csv(spectral_metrics, "18_spectral_metrics.csv")
spectral_metrics


In [ ]:
eig_df = pd.DataFrame({
    "index": np.arange(len(eigvals)),
    "real": np.real(eigvals),
    "imag": np.imag(eigvals),
    "abs": np.abs(eigvals),
})
save_csv(eig_df, "18_P_eigenvalues.csv")

plt.figure(figsize=(8, 7))
plt.scatter(np.real(eigvals), np.imag(eigvals), s=90)
unit = plt.Circle((0, 0), 1, fill=False, linestyle="--", alpha=0.5)
plt.gca().add_patch(unit)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("real")
plt.ylabel("imaginary")
plt.title(f"Eigenvalue spectrum of P (gap={spectral_gap:.4f})")
plt.axis("equal")
savefig("18_P_eigenvalue_spectrum.png")


In [ ]:
plt.figure(figsize=(10, 6))
for k in range(min(4, eigvecs.shape[1])):
    vec = np.real(eigvecs[:, k])
    plt.plot(RESIDUES, vec, marker="o", label=f"eigenvector {k}, |lambda|={lambda_abs[k]:.3f}")
plt.xlabel("state residue mod 30")
plt.ylabel("real eigenvector component")
plt.title("Leading eigenvectors of P")
plt.legend()
savefig("18_P_leading_eigenvectors.png")

plt.figure(figsize=(8, 6))
plt.bar([str(r) for r in RESIDUES], stationary)
plt.xlabel("state residue mod 30")
plt.ylabel("stationary probability")
plt.title("Stationary distribution implied by P")
savefig("18_stationary_distribution.png")

save_csv(pd.DataFrame({"residue": RESIDUES, "stationary_probability": stationary}), "18_stationary_distribution.csv")


## 5. Singular-value decomposition of the memory operator

We decompose `Delta = U S Vt`. If memory is low-dimensional, the first few singular values capture most of the Frobenius energy.

In [ ]:
U, S, Vt = np.linalg.svd(Delta, full_matrices=False)
energy = S**2
energy_share = energy / energy.sum() if energy.sum() > 0 else np.zeros_like(energy)
cumulative_energy = np.cumsum(energy_share)
effective_rank_entropy = float(np.exp(-np.sum(energy_share * np.log(energy_share + 1e-15))))
best_k_90 = int(np.searchsorted(cumulative_energy, 0.90) + 1)
best_k_95 = int(np.searchsorted(cumulative_energy, 0.95) + 1)

sv_df = pd.DataFrame({
    "mode": np.arange(1, len(S) + 1),
    "singular_value": S,
    "energy_share": energy_share,
    "cumulative_energy": cumulative_energy,
})
save_csv(sv_df, "18_delta_singular_values.csv")

summary.update({
    "delta_frobenius_norm": float(np.linalg.norm(Delta, ord="fro")),
    "delta_effective_rank_entropy": effective_rank_entropy,
    "best_k_90_energy": best_k_90,
    "best_k_95_energy": best_k_95,
    "spectral_gap": spectral_gap,
    "mixing_proxy_1_over_gap": mixing_proxy,
})
sv_df


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(sv_df["mode"], sv_df["singular_value"], marker="o", label="singular value")
plt.xlabel("mode")
plt.ylabel("singular value")
plt.title("Memory operator singular values")
plt.legend()
savefig("18_delta_singular_value_spectrum.png")

plt.figure(figsize=(9, 6))
plt.plot(sv_df["mode"], sv_df["cumulative_energy"], marker="o", label="cumulative energy")
plt.axhline(0.90, linestyle="--", label="90%")
plt.axhline(0.95, linestyle="--", label="95%")
plt.xlabel("rank k")
plt.ylabel("cumulative energy share")
plt.title("Cumulative memory energy captured by low-rank modes")
plt.legend()
savefig("18_delta_cumulative_energy.png")


In [ ]:
for idx in range(min(3, len(S))):
    mode = S[idx] * np.outer(U[:, idx], Vt[idx, :])
    heatmap(mode, f"Memory mode {idx+1} (singular value={S[idx]:.5f})", "delta contribution", f"18_delta_memory_mode_{idx+1}.png", center_zero=True)


## 6. Low-rank reconstruction of memory

Compute `M_k = sum_i^k S_i U_i V_i^T` and compare it to the full memory operator `Delta`.

In [ ]:
def low_rank_delta(k):
    return (U[:, :k] * S[:k]) @ Vt[:k, :]

rows = []
base_norm = np.linalg.norm(Delta, ord="fro")
for k in range(1, N_STATES + 1):
    Mk = low_rank_delta(k)
    err_fro = np.linalg.norm(Delta - Mk, ord="fro")
    rows.append({
        "rank_k": k,
        "frobenius_error": err_fro,
        "relative_frobenius_error": err_fro / base_norm if base_norm > 0 else np.nan,
        "energy_captured": cumulative_energy[k-1],
    })
low_rank_df = pd.DataFrame(rows)
save_csv(low_rank_df, "18_low_rank_reconstruction_errors.csv")
low_rank_df


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(low_rank_df["rank_k"], low_rank_df["relative_frobenius_error"], marker="o", label="relative error")
plt.xlabel("rank k")
plt.ylabel("relative Frobenius error")
plt.title("Low-rank memory reconstruction error")
plt.legend()
savefig("18_low_rank_error_curve.png")

for k in [1, 2, 3, best_k_90]:
    if 1 <= k <= N_STATES:
        heatmap(low_rank_delta(k), f"Low-rank memory reconstruction M_{k}", "delta contribution", f"18_delta_reconstructed_rank_{k}.png", center_zero=True)


## 7. Corrected two-step operators

Compare Markov baseline `P2_markov`, corrected model `P2_markov + M_k`, and empirical target `P2_empirical`.

Corrected operators are clipped at zero and row-normalized before probability metrics are computed.

In [ ]:
def normalize_operator(M):
    X = np.maximum(np.asarray(M, dtype=float), 0.0)
    return row_normalize(X)

def js_divergence_matrix(A, B, eps=1e-12):
    A = normalize_operator(A) + eps
    B = normalize_operator(B) + eps
    A = A / A.sum(axis=1, keepdims=True)
    B = B / B.sum(axis=1, keepdims=True)
    M = 0.5 * (A + B)
    kl_a = np.sum(A * np.log(A / M), axis=1)
    kl_b = np.sum(B * np.log(B / M), axis=1)
    return float(np.mean(0.5 * (kl_a + kl_b)))

def l1_error(A, B):
    return float(np.mean(np.sum(np.abs(A - B), axis=1)))

def l2_error(A, B):
    return float(np.linalg.norm(A - B, ord="fro"))

model_rows = []
models = {"markov_P2": P2_markov}
for k in range(1, N_STATES + 1):
    models[f"corrected_rank_{k}"] = normalize_operator(P2_markov + low_rank_delta(k))

for model_name, M in models.items():
    model_rows.append({
        "model": model_name,
        "rank_k": 0 if model_name == "markov_P2" else int(model_name.split("_")[-1]),
        "l1_error": l1_error(P2_empirical, M),
        "l2_error": l2_error(P2_empirical, M),
        "js_divergence": js_divergence_matrix(P2_empirical, M),
    })
model_df = pd.DataFrame(model_rows)
save_csv(model_df, "18_model_comparison_metrics.csv")
model_df


In [ ]:
plt.figure(figsize=(10, 6))
for metric in ["l1_error", "l2_error", "js_divergence"]:
    plt.plot(model_df["rank_k"], model_df[metric], marker="o", label=metric)
plt.xlabel("rank k (0 = Markov P²)")
plt.ylabel("error / divergence")
plt.title("Predictive error versus memory rank")
plt.legend()
savefig("18_prediction_error_vs_rank.png")

plot_df = model_df.copy()
plot_df["model_label"] = plot_df["model"].str.replace("corrected_", "", regex=False)
plt.figure(figsize=(11, 6))
plt.bar(plot_df["model_label"], plot_df["l2_error"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("L2/Frobenius error")
plt.title("Two-step operator prediction error")
savefig("18_model_comparison_l2_errors.png")


In [ ]:
chosen_k = best_k_90
P2_corrected = normalize_operator(P2_markov + low_rank_delta(chosen_k))
corrected_delta = P2_empirical - P2_corrected

save_csv(matrix_df(P2_corrected), f"18_corrected_two_step_operator_rank_{chosen_k}.csv")
save_csv(matrix_df(corrected_delta), f"18_corrected_residual_delta_rank_{chosen_k}.csv")

heatmap(P2_corrected, f"Corrected two-step operator P² + M_{chosen_k}", "probability", f"18_corrected_two_step_operator_rank_{chosen_k}.png")
heatmap(corrected_delta, f"Residual after rank-{chosen_k} correction", "remaining delta", f"18_corrected_vs_empirical_delta_rank_{chosen_k}.png", center_zero=True)


## 8. Mode interpretation by residue

The leading singular vectors define row-side and column-side memory biases.

In [ ]:
mode_rows = []
for mode_idx in range(min(4, len(S))):
    for side, vec in [("left_current_state", U[:, mode_idx]), ("right_two_step_state", Vt[mode_idx, :])]:
        for residue, value in zip(RESIDUES, vec):
            mode_rows.append({
                "mode": mode_idx + 1,
                "side": side,
                "residue": int(residue),
                "component": float(value),
                "abs_component": float(abs(value)),
            })
mode_projection_df = pd.DataFrame(mode_rows)
save_csv(mode_projection_df, "18_mode_residue_projection.csv")
mode_projection_df.head()


In [ ]:
for mode_idx in range(min(3, len(S))):
    plt.figure(figsize=(10, 6))
    width = 0.35
    x = np.arange(N_STATES)
    plt.bar(x - width/2, U[:, mode_idx], width=width, label="current-state vector U")
    plt.bar(x + width/2, Vt[mode_idx, :], width=width, label="two-step-state vector V")
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x, RESIDUES)
    plt.xlabel("residue mod 30")
    plt.ylabel("singular-vector component")
    plt.title(f"Residue projection for memory mode {mode_idx+1}")
    plt.legend()
    savefig(f"18_mode_{mode_idx+1}_residue_projection.png")


## 9. Summary tables

Notebook 18 generates a compact interpretation row to carry forward into Notebook 19.

In [ ]:
best_row = model_df.sort_values("l2_error").iloc[0].to_dict()
markov_row = model_df[model_df["model"] == "markov_P2"].iloc[0]
improvement_l2 = float((markov_row["l2_error"] - best_row["l2_error"]) / markov_row["l2_error"]) if markov_row["l2_error"] else np.nan

interpretation = pd.DataFrame([{
    "notebook": NOTEBOOK_ID,
    "limit": LIMIT,
    "state_count": N_STATES,
    "spectral_gap": spectral_gap,
    "mixing_proxy_1_over_gap": mixing_proxy,
    "delta_frobenius_norm": summary["delta_frobenius_norm"],
    "delta_effective_rank_entropy": effective_rank_entropy,
    "rank_for_90pct_memory_energy": best_k_90,
    "rank_for_95pct_memory_energy": best_k_95,
    "best_model": best_row["model"],
    "best_model_l2_error": best_row["l2_error"],
    "markov_l2_error": float(markov_row["l2_error"]),
    "relative_l2_improvement_vs_markov": improvement_l2,
}])
save_csv(interpretation, "18_interpretation_summary.csv")
interpretation


## 10. Write summary markdown

This markdown file is intended for quick repo review and paper-note extraction.

In [ ]:
summary_md = f"""# Notebook 18 — Spectral memory decomposition and low-rank correction

## Purpose

Notebook 18 tests whether the two-step residue memory found in Notebook 17 is compressible as a low-rank correction to the first-order Markov baseline.

## Core equation

```text
P2_empirical = P^2 + Delta ≈ P^2 + M_k
```

## Key results

- Prime limit: `{LIMIT:,}`
- Used primes after 2,3,5 exclusion: `{len(primes):,}`
- First-order state count: `{N_STATES}`
- Spectral gap proxy of P: `{spectral_gap:.6f}`
- Mixing proxy 1/gap: `{mixing_proxy:.3f}`
- Memory operator Frobenius norm: `{summary['delta_frobenius_norm']:.6f}`
- Effective memory rank: `{effective_rank_entropy:.3f}`
- Rank for 90% memory energy: `{best_k_90}`
- Rank for 95% memory energy: `{best_k_95}`
- Best corrected model: `{best_row['model']}`
- Markov L2 error: `{float(markov_row['l2_error']):.6f}`
- Best-model L2 error: `{float(best_row['l2_error']):.6f}`
- Relative L2 improvement vs Markov: `{improvement_l2:.6%}`

## Interpretation

The empirical two-step operator is not fully explained by the first-order Markov baseline. The deviation operator Delta is structured and can be decomposed into singular modes. If the cumulative singular-value energy concentrates in a small number of modes, the memory effect is low-dimensional rather than random noise.

## Next notebook

Notebook 19 should use the corrected operator as a generative model and compare simulated residue trajectories against observed transition, memory, entropy, and distributional metrics.
"""

save_text(summary_md, "18_summary.md")
print(summary_md)


## 11. Manifest and output zip

Template-standard finalization: write manifest, then create a zip containing all generated outputs. The download cell is optional and works in Google Colab after the zip exists.

In [ ]:
manifest_df = pd.DataFrame(manifest).drop_duplicates("file").sort_values(["type", "file"])
save_csv(manifest_df, "18_outputs_manifest.csv")
manifest_df


In [ ]:
# Create outputs bundle before the optional Colab download cell.
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in pd.DataFrame(manifest).drop_duplicates("file")["file"]:
        path = OUT / item
        if path.exists():
            zf.write(path, arcname=path.name)

print(f"Created {OUTPUT_ZIP}")
print(f"Files included: {len(pd.DataFrame(manifest).drop_duplicates('file'))}")


In [ ]:
# Optional: download outputs bundle (template standard)
from google.colab import files
files.download("18_spectral_memory_decomposition_and_low_rank_correction_outputs.zip")
